# Dice

Simple model of a dice game, no cards.

In [1]:
import numpy as np
from random import choice
import time
from collections import Counter

In [56]:
def calc_score(dice):
    TRIPLE_VALUES = [1000, 200, 300, 400, 500, 600]
    SINGLE_VALUES = [100, 0, 0, 0, 50, 0]

    # Extract the values from the dice
    values = [die.value for die in dice]
    dice = np.array(values)
    num_dice = len(dice)
    score = 0

    # Check for a straight
    if num_dice == 6 and (dice == [1, 2, 3, 4, 5, 6]).all():
        return 1500

    # Count of each number's occurence
    counts = np.array([Counter(dice)[k] for k in range(1, 7)])

    # Check for three of a kind
    score = 0
    while (counts >= 3).any():
        score += np.dot(counts >= 3, TRIPLE_VALUES)
        num_dice -= 3 * sum(counts >= 3)
        counts[counts >= 3] = counts[counts >= 3] - 3

    # Check for singles (1's and 5's)
    score += np.dot(counts, SINGLE_VALUES)
    num_dice -= np.dot(counts, [1, 0, 0, 0, 1, 0])

    # Determine if this roll is a fill or bust
    return score

In [241]:
class Die(object):
    
    POSSIBLE_DICE_VALUES = list(range(1, 7))
    
    def __init__(self):
        self.value = choice(self.POSSIBLE_DICE_VALUES)
        self.is_held = False
        self.is_banked = False
        
    def __lt__(self, die):
        return self.value < die.value

    def __gt__(self, die):
        return self.value > die.value

    def __repr__(self):
        return f'*{self.value}*' if self.is_held else f'<{self.value}>'

class Player(object):
    
    def __init__(self, name, score=0):
        self.name = name
        self.score = score
        
    def __repr__(self):
        return f'<Player: {self.name}, Score: {self.score}>'

In [319]:
class Game(object):
    
    def __init__(self, players):
        self.players = players
        self.num_players = len(self.players)
        self.current_player_index = choice(range(self.num_players))
        self.max_score = 1000
        self.new_turn()
        self.print_new_game()

    def print_new_game(self):
        print('New game!')
        print(f'Players: {", ".join([player.name for player in self.players])}')
        print(f'Max score: {self.max_score}')
        print(f'Up first: {self.current_player.name}')
        
    def increment_current_player_index(self):
        self.current_player_index = (self.current_player_index + 1) % self.num_players
        
    def new_turn(self):
        self.increment_current_player_index()
        self.turn_score = 0
        self.num_dice = 6
        self.dice = []
        self.held_dice = []
        self.state = 'rolling'
        
    @property
    def current_player(self):
        return self.players[self.current_player_index]
        
    @property
    def somebody_has_won(self):
        return any(player.score >= self.max_score for player in self.players)
    
    @property
    def game_state(self):
        state = {'current_player': self.current_player,
                 'dice': self.dice,
                 'held_dice': self.held_dice,
                 'state': self.state,
                 'valid_actions': self.valid_actions,
                 'turn_score': self.turn_score}
        return state

    def _is_valid_action(self, action):
        if self.somebody_has_won:
            print('Game over! Somebody won.')
            return dict(zip(['roll', 'hold', 'stop'], [False, False, False]))
        return {'roll': ((self.state == 'rolling') or
                         (self.state == 'holding' and self.dice_are_held) or
                         (self.state == 'filled')),
                'stop': (self.state == 'holding' and self.dice_are_held or self.state == 'filled'),
                'hold': (self.state == 'holding') and self.state != 'filled'}[action]

    @property
    def valid_actions(self):
        actions = ['roll', 'hold', 'stop']
        return {action: self._is_valid_action(action) for action in actions}
    
    def action_is_valid(self, action):
        valid = ', '.join(key for key, val in self.valid_actions.items() if val)
        msg = f'"{action}" is not a valid action currently. Valid actions: {valid}'
        assert self.valid_actions[action], msg

    @property
    def valid_holds(self):
        counts = Counter([die.value for die in self.dice])
        holds = {}
        for idx, die in enumerate(self.dice):
            holds[idx] = ((counts[die.value] >= 3) or (die.value in [1, 5]))
        return holds
    
    @property
    def dice_are_held(self):
        return any(die.is_held for die in self.dice)
    
    def roll(self):
        self.action_is_valid('roll')
        self.dice = sorted([Die() for die in range(self.num_dice)])
        print(self.dice)
        score = calc_score(self.dice)
        if score == 0:
            print('Bust!')
            self.state = 'busted'
            return self.end_turn()
        self.state = 'holding'
    
    def hold(self, idxs=[]):
        self.action_is_valid('hold')
        if all(self.valid_holds.values()):
            print('Fill!')
            self.turn_score = calc_score(self.dice)
            self.state = 'filled'
        idxs = idxs if idxs else list(range(len(self.dice)))
        for idx in idxs:
            if self.valid_holds[idx]:
                self.dice[idx].is_held = self.dice[idx].is_held == False
                if self.dice[idx] in self.held_dice:
                    self.held_dice.remove(self.dice[idx])
                    self.num_dice += 1
                else:
                    self.held_dice.append(self.dice[idx])
                    self.num_dice -= 1
        print(self.dice)
        return
    
    def stop(self):
        self.action_is_valid('stop')
        self.end_turn()
        
    def end_turn(self):
        if self.state == 'busted':
            self.held_dice = []
        self.turn_score = calc_score(self.held_dice)
        print(f"{self.current_player.name}'s score for this turn: {self.turn_score}")
        self.current_player.score += self.turn_score
        if self.current_player.score >= self.max_score:
            print(f'{self.current_player.name} wins! Score: {self.current_player.score}')
            player_scores = [player.score for player in self.players]
            print(f'Final Scores: {player_scores}')
            return
        
        self.new_turn()
    
    def take_turn(self):
        print(f'current_player: {self.current_player}')
        self.new_turn()
        # 1) Wait for roll
        # 2) Check for fill/bust
        # 3) Wait for holds
        # 4) Wait for stop
        # 5) If not stop, remove held dice, go back to 1)

In [320]:
p1 = Player('John')
p2 = Player('Anna')
game = Game(players=[p1, p2])
game

New game!
Players: John, Anna
Max score: 1000
Up first: Anna


In [335]:
game.game_state

Game over! Somebody won.
Game over! Somebody won.
Game over! Somebody won.


{'current_player': <Player: John, Score: 1100>,
 'dice': [*1*, *1*, <4>],
 'held_dice': [*1*, *5*, *5*, *1*, *1*],
 'state': 'holding',
 'valid_actions': {'roll': {'roll': False, 'hold': False, 'stop': False},
  'hold': {'roll': False, 'hold': False, 'stop': False},
  'stop': {'roll': False, 'hold': False, 'stop': False}},
 'turn_score': 1100}

In [332]:
game.hold()

[*1*, *1*, <4>]


In [331]:
game.roll()

[<1>, <1>, <4>]


In [334]:
game.stop()

John's score for this turn: 1100
John wins! Score: 1100
Final Scores: [1100, 450]
